# Init

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

cust_info_df = spark.table('bike_lakehouse.bronze.cust_info_bronze')

cust_info_df.display()

In [0]:
COLUMNS_NAME = {
        "cst_id": "customer_id",
        "cst_key": "customer_key",
        "cst_firstname": "firstname",
        "cst_lastname": "lastname",
        "cst_marital_status": "marital_status",
        "cst_gndr": "gender",
        "cst_create_date": "create_date"
    }


## Rename Columns

In [0]:
customer_information = cust_info_df
for old_name, new_name in COLUMNS_NAME.items():
  customer_information = customer_information.withColumnRenamed(old_name, new_name)
customer_information.display()
# customer_information = (cust_info
#                         .withColumnRenamed('cst_id', 'customer_id')
#                         .withColumnRenamed('cst_key', 'key')
#                         .withColumnRenamed('cst_firstname', 'firstname')
#                         .withColumnRenamed('cst_lastname', 'lastname')
#                         .withColumnRenamed('cst_marital_status', 'marital_status')
#                         .withColumnRenamed('cst_gndr', 'gender')
#                         .withColumnRenamed('cst_create_date', 'create_date'))
# display(customer_information)

## Trim all string

In [0]:
for field in customer_information.schema.fields:
    if isinstance(field.dataType, StringType):
        customer_information = customer_information.withColumn(field.name, trim(col(field.name)))

customer_information.show()

In [0]:
display(customer_information.select("gender").distinct())

## Standadize column values Eg from 'M' to 'Male'

In [0]:
customer_information = customer_information.withColumn(
    "marital_status",
    when(col("marital_status") == "M", "Married")
    .when(col("marital_status") == "S", "Single")
    .otherwise("n/a")
).withColumn(
    "gender",
    when(col("gender") == "M", "Male")
    .when(col("gender") == "F", "Female")
    .otherwise("n/a")
)

## Remove duplicate from the customer_id

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy(col("create_date").desc())
customer_information_unique = customer_information.withColumn(
    "rank_latest_data",
    row_number().over(window_spec)
).filter(col("rank_latest_data") == 1).filter(col("customer_id").isNotNull()).drop("rank_latest_data")

display(customer_information_unique.select("customer_id").distinct())

# Write to silver Layer

In [0]:
customer_information_unique.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bike_lakehouse.silver.customer_information_silver")